# Data quality — detect and report

Runs after the pipeline and checks Bronze, Silver and Gold in one pass.

**This notebook never changes or blocks data.** It measures and records. That is a
deliberate choice with a real trade-off: bad rows reach Gold, and people find out
from an alert rather than from a missing row. What makes it acceptable here is that
Silver is *merge*-based — a broken or empty extract merges nothing rather than
wiping the table, and Gold rebuilds from a Silver that still holds good data.

Two kinds of check:

* **rules** — column-level expectations per table (`quality/config/rules.json`)
* **reconciliations** — one SQL statement returning a single `passed` boolean,
  for the cross-table promises: line totals roll up to headers, no orphan keys.

Everything lands in `ops.data_quality_results`. Alert on it:

```sql
SELECT count(*) FROM ops.data_quality_results
WHERE run_date = current_date() AND severity = 'error' AND NOT passed;
```

In [ ]:
import sys
from datetime import date
from pathlib import Path

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "common_utils").is_dir():
        sys.path.insert(0, str(candidate))
        break

from common_utils.logger import get_logger, log_info, log_warning
from common_utils.quality import RuleResult, ensure_results_table, evaluate_rules, failing_rows, persist_results
from common_utils.settings import load_json, parse_run_date

In [ ]:
dbutils.widgets.text("config_path", "quality/config/rules.json")
dbutils.widgets.text("catalog", "retaildataplatform")
dbutils.widgets.text("bronze_schema", "bronze")
dbutils.widgets.text("silver_schema", "silver")
dbutils.widgets.text("gold_schema", "gold")
dbutils.widgets.text("ops_schema", "ops")
dbutils.widgets.text("run_date", date.today().isoformat())

catalog = dbutils.widgets.get("catalog")
schemas = {
    "bronze": dbutils.widgets.get("bronze_schema"),
    "silver": dbutils.widgets.get("silver_schema"),
    "gold": dbutils.widgets.get("gold_schema"),
}
ops_schema = dbutils.widgets.get("ops_schema")
run_date = parse_run_date(dbutils.widgets.get("run_date"))
config = load_json(dbutils.widgets.get("config_path"))
logger = get_logger("quality")

ensure_results_table(spark, catalog, ops_schema)

## 1. Column rules
Bronze is checked for *this run's* rows only; Silver and Gold hold merged state,
so they are checked whole.

In [ ]:
summary = []
for check in config["checks"]:
    layer, table = check["layer"], check["table"]
    full_name = f"{catalog}.{schemas[layer]}.{table}"
    if not spark.catalog.tableExists(full_name):
        log_warning(logger, "table missing, skipped", table=full_name)
        continue

    df = spark.table(full_name)
    if layer == "bronze":
        df = df.filter(f"_load_date = '{run_date}'")

    for rule in check["rules"]:
        scoped = df.filter(rule["filter"]) if rule.get("filter") else df
        results = evaluate_rules(scoped, [rule])
        persist_results(spark, catalog, ops_schema, run_date, layer, table, results)
        result = results[0]
        summary.append((layer, table, rule["name"], rule.get("severity", "error"), result.rows_checked, result.failed_rows, result.passed))
        if not result.passed:
            log_warning(
                logger,
                "rule failed",
                layer=layer,
                table=table,
                rule=rule["name"],
                severity=rule.get("severity", "error"),
                failed_rows=result.failed_rows,
            )

## 2. Reconciliations
Cross-table promises. A rule can tell you a column is null; only a reconciliation
can tell you the star has quietly lost revenue.

In [ ]:
def render(sql: str) -> str:
    return (
        sql.replace("${catalog}", catalog)
        .replace("${bronze}", schemas["bronze"])
        .replace("${silver}", schemas["silver"])
        .replace("${gold}", schemas["gold"])
    )


for check in config.get("reconciliations", []):
    try:
        passed = bool(spark.sql(render(check["sql"])).first()[0])
    except Exception as exc:  # noqa: BLE001 - a broken check must not stop the others
        log_warning(logger, "reconciliation could not run", name=check["name"], error=str(exc).splitlines()[0][:200])
        continue

    rule = {"name": check["name"], "type": "expression", "severity": check.get("severity", "error"), "description": check.get("description")}
    persist_results(spark, catalog, ops_schema, run_date, "gold", "reconciliation", [RuleResult(rule, 1, 0 if passed else 1)])
    summary.append(("gold", "reconciliation", check["name"], rule["severity"], 1, 0 if passed else 1, passed))
    if not passed:
        log_warning(logger, "reconciliation failed", name=check["name"])

## 3. Report

In [ ]:
report = spark.createDataFrame(summary, "layer string, table_name string, rule_name string, severity string, rows_checked long, failed_rows long, passed boolean")
errors = report.filter("NOT passed AND severity = 'error'")
warnings = report.filter("NOT passed AND severity = 'warn'")

print(f"run_date={run_date}  checks={report.count()}  errors={errors.count()}  warnings={warnings.count()}")
log_info(logger, "quality run finished", run_date=run_date, checks=report.count(), errors=errors.count(), warnings=warnings.count())

display(report.filter("NOT passed").orderBy("severity", "layer", "table_name"))

## 4. Investigate
For any failing row-level rule, look at the offending rows. Change the table and
rule name below when you are chasing something specific.

In [ ]:
if errors.count() > 0:
    first = errors.first()
    table_check = next((c for c in config["checks"] if c["table"] == first["table_name"] and c["layer"] == first["layer"]), None)
    if table_check:
        source = spark.table(f"{catalog}.{schemas[first['layer']]}.{first['table_name']}")
        if first["layer"] == "bronze":
            source = source.filter(f"_load_date = '{run_date}'")
        display(failing_rows(source, [r for r in table_check["rules"] if r["name"] == first["rule_name"]]))
else:
    print("No error-severity failures.")